## Exercise 2

**This class is Homework 1, questions 2 and 3.** By the end you will have coded the
three information criteria yourself and compared them with the top-down sequence.

Note the assignment rule: you are **not allowed to use built-in functions** for these,
so everything here is written from scratch with `numpy`.

#### Important linear algebra operations

The following goes through the most popular linear algebra operations that we will use in the exercises.

In [1]:
import numpy as np

A = np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9]])
B = np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9]])

C = np.array([[5, 6, 7], 
            [8, 3, 1], 
            [6, 3, 8]])

Tranpose of a vector: $B^\prime $

In [2]:
B_transpose = B.T
B_transpose

array([[1],
       [2],
       [3],
       [4],
       [5],
       [6],
       [7],
       [8],
       [9]])

Product of two vectors/matrices: $A \cdot B^\prime$

In [3]:
AB_transpose = A @ B_transpose
AB_transpose

array([[285]])

Inverse of a matrix: $C^{-1}$

In [4]:
C_inv = np.linalg.inv(C)
C_inv

array([[-0.10447761,  0.13432836,  0.07462687],
       [ 0.28855721,  0.00995025, -0.25373134],
       [-0.02985075, -0.10447761,  0.1641791 ]])

Determinant of a matrix: $\det(C)$

In [5]:
C_det = np.linalg.det(C)
C_det

np.float64(-201.0)

Trace of a matrix: $tr(C)$

In [6]:
C_trace = np.trace(C)
C_trace

np.int64(16)

#### The information criteria

There are three common information criteria known as The Akaike Information Criterion (AIC), Hannan-Quin Criterion (HQC) and the Schwarz Information Criterion (SIC). 

$m$ is the amount of Lags in the model.

The AIC is given as:

\begin{equation*}
    \text{AIC}(m) = \log\left(\det\left(\widetilde{\Sigma}_{u}(m)\right)\right) + \frac{2}{T}\left(mK^{2} + K\right)
\end{equation*}

where $\widetilde{\Sigma}_{u}(m)=T^{-1}\sum_{t=1}^{T}\hat{u}_{t}\hat{u}_{t}^{\prime}$ is the covariance matrix estimator for the least square residual, $\hat{u}_{t}$, at a given lag order, $m$. $K$ is the number of equations in the model and $T$ is the number of observations used. 

The HQC is given as:


\begin{equation*}
    \text{HQC}(m)=\log\left(\det\left(\widetilde{\Sigma}_{u}(m)\right)\right)+\frac{2\log(\log(T))}{T}\left(mK^{2}+K\right)
\end{equation*}

The SIC is given as:

\begin{equation*}
    \text{SIC}(m)=\log\left(\det\left(\widetilde{\Sigma}_{u}(m)\right)\right)+\frac{\log(T)}{T}\left(mK^{2}+K\right)
\end{equation*}

Your goal is to implement each information criteria and report the one number of lags that produces the smalles information criteria.

1. The following code generates an VAR(1) model with 2 variables. Make sure you understand how the VAR(1) process is generated. 

In [7]:
%reset -f

import numpy as np
import pandas as pd

pd.set_option('display.float_format', '{:.4f}'.format)

np.random.seed(1)

# Parameters
N = 100  # Number of observations
rho1 = np.array([[0.8, 0],  # Autoregressive coefficient for lag 1
                 [0, 0.8]])

alpha = np.array([0, 0])  # Mean of the VAR process

sigma = np.array([[1, 0],  # Covariance matrix of white noise
                  [0, 1]])

# Pre-allocate the array for the VAR(2) process
y = np.zeros((2, N))

# Generate white noise
epsilon = np.random.multivariate_normal([0, 0], sigma, N).T

# Generate the VAR(2) process
for t in range(1, N):
    y[:, t] = alpha + rho1 @ y[:, t-1]  + epsilon[:, t]

y = y.T


2. Fill out the missing parts of the function "pfind(y, max)" 


Hint: Remember that if we estimate the companion form of the VAR(m) model, $Y_{t}=\beta X_{t}+u_{t}$, then we have that $u_{t}=Y_{t}-\beta X_{t}$, $\beta=\left(YX^{\prime}\right) \left(XX^{\prime}\right)^{-1}$ and $\Sigma_{u}=\frac{1}{T}uu^{\prime}$

In [8]:
def pfind(y, pmax):
    t, K = y.shape

    # Construct regressor matrix and dependent variable
    XMAX = np.ones((1, t - pmax))
    
    for i in range(1, pmax + 1):
        XMAX = np.vstack([XMAX, y[pmax - i:t - i, :].T])

    Y = y[pmax:t, :].T

    aiccrit = np.zeros((pmax + 1, 1))
    hqccrit = np.zeros((pmax + 1, 1))
    siccrit = np.zeros((pmax + 1, 1))

    # Evaluate criterion for p = 0,...,pmax
    for j in range(0, pmax + 1):
        m = j
        T = t - pmax
        X = XMAX[:j * K + 1, :]

        B = Y @ X.T @ np.linalg.inv(X @ X.T)

        SIGMA = (Y - B @ X) @ (Y - B @ X).T / T

        aiccrit[j] = np.log(np.linalg.det(SIGMA)) + 2 / T * (m * K ** 2 + K)       # AIC value
        hqccrit[j] = np.log(np.linalg.det(SIGMA)) + 2 * np.log(np.log(T)) / T * (m * K ** 2 + K)  # HQC value
        siccrit[j] = np.log(np.linalg.det(SIGMA)) + np.log(T) / T * (m * K ** 2 + K)  # SIC value

    # Rank models for p = 0,1,2,...,pmax
    aichat = np.argmin(aiccrit)
    hqchat = np.argmin(hqccrit)
    sichat = np.argmin(siccrit)

    infomat = np.hstack([siccrit, hqccrit, aiccrit])
    m = np.arange(0, pmax + 1).reshape(-1, 1)
    imat = np.hstack([m, infomat])

    LagInformationValue = pd.DataFrame(imat, columns=['Lag', 'SIC', 'HQ', 'AIC'])
    OptimalLag = pd.DataFrame([[sichat, hqchat, aichat]], columns=['SIC', 'HQ', 'AIC'])

    return LagInformationValue, OptimalLag


LagInformationValue, OptimalLag = pfind(y,12)


In [9]:
LagInformationValue

,Lag,SIC,HQ,AIC
0,0.0000,1.3068,1.2732,1.2505
1,1.0000,-0.2371,-0.3379,-0.4060
2,2.0000,-0.1492,-0.3173,-0.4307
3,3.0000,0.0280,-0.2073,-0.3661
4,4.0000,0.2000,-0.1026,-0.3067
5,5.0000,0.3634,-0.0064,-0.2559
6,6.0000,0.4971,0.0600,-0.2348
7,7.0000,0.6766,0.1723,-0.1680
8,8.0000,0.7884,0.2168,-0.1688
9,9.0000,0.9508,0.3120,-0.1190


In [10]:
OptimalLag

,SIC,HQ,AIC
0,1,1,2


If implemented correctly, the function should output the following values:

| **Lag** | **SIC**  | **HQ**   | **AIC**  |
|---------|----------|----------|----------|
| 0.0000  | 1.3068   | 1.2732   | 1.2505   |
| 1.0000  | -0.2371  | -0.3379  | -0.4060  |
| 2.0000  | -0.1492  | -0.3173  | -0.4307  |
| 3.0000  | 0.0280   | -0.2073  | -0.3661  |
| 4.0000  | 0.2000   | -0.1026  | -0.3067  |
| 5.0000  | 0.3634   | -0.0064  | -0.2559  |
| 6.0000  | 0.4971   | 0.0600   | -0.2348  |
| 7.0000  | 0.6766   | 0.1723   | -0.1680  |
| 8.0000  | 0.7884   | 0.2168   | -0.1688  |
| 9.0000  | 0.9508   | 0.3120   | -0.1190  |
| 10.0000 | 1.1389   | 0.4329   | -0.0435  |
| 11.0000 | 1.2849   | 0.5116   | -0.0101  |
| 12.0000 | 1.4125   | 0.5720   | 0.0049   |


3. Consider what the maximum lag length should be. What is the tradeoff when setting a higher maximum lag length?

4. Try changing the seed. Does the optimal information criteria always give the same conclusion? What if the number of observations are changed?

5. Read page 58-60 in Kilian and Lütkepohl (2017). How consistent is the information criteria for the true lag length and how consistent are they in small samples?

6. Expand the VAR(1) process to a VAR(2) process by making $\rho_{2}=\left[\begin{array}{cc}
0.15 & 0\\
0 & 0.15
\end{array}\right]$. Make sure the code still works. Are the conclusions above still the same?

**Answer to question 3.** `pmax` has to be chosen *before* you look at any criterion, and it
is not a free parameter:

- Every model in the table is estimated on the **same** $T - p_{max}$ observations, so raising
  `pmax` throws data away from *every* row, including the short models you might end up
  choosing. With `pmax = 12` and 100 observations you compare all models on 88.
- The top of the table also becomes badly over-parameterised: at $m = 12$ with $K = 2$ you
  estimate 49 coefficients per equation.
- Set it too low and you may truncate below the true order and never see it.

The practical rule is to set `pmax` from the data frequency: **4 (or 8) for quarterly data,
12 for monthly**. And if a criterion ends up selecting `pmax` itself, you have not found a
minimum — you have hit the edge of the table, and should re-run with a larger `pmax` or
accept that the sample cannot support that many lags.

In [11]:
# Question 4: does the conclusion depend on the seed, and on the sample size?

def simulate(seed, N, p_true=1):
    """The same DGP as above, wrapped so we can vary the seed and the length."""
    np.random.seed(seed)
    rho1 = np.array([[0.8, 0], [0, 0.8]])
    rho2 = np.array([[0.15, 0], [0, 0.15]])
    y = np.zeros((2, N))
    epsilon = np.random.multivariate_normal([0, 0], np.eye(2), N).T
    for t in range(2, N):
        y[:, t] = rho1 @ y[:, t - 1] + epsilon[:, t]
        if p_true == 2:
            y[:, t] += rho2 @ y[:, t - 2]
    return y.T

rows = []
for s in range(1, 9):                      # vary the seed, hold N = 100
    opt = pfind(simulate(s, 100), 12)[1]
    rows.append([f"seed {s}, N=100"] + opt.iloc[0].tolist())
for N in (50, 100, 200, 500):              # vary N, hold the seed at 1
    opt = pfind(simulate(1, N), 12)[1]
    rows.append([f"seed 1, N={N}"] + opt.iloc[0].tolist())

pd.DataFrame(rows, columns=["setting", "SIC", "HQ", "AIC"])

,setting,SIC,HQ,AIC
0,"seed 1, N=100",1,1,2
1,"seed 2, N=100",1,1,1
2,"seed 3, N=100",1,1,1
3,"seed 4, N=100",1,1,1
4,"seed 5, N=100",1,1,1
5,"seed 6, N=100",1,1,6
6,"seed 7, N=100",1,1,1
7,"seed 8, N=100",1,1,1
8,"seed 1, N=50",1,1,3
9,"seed 1, N=100",1,1,2


**Answer to question 4.** No — the criteria are statistics, so they are random variables, and a
single seed tells you very little.

- **Changing the seed:** SIC and HQ return 1 for every seed above, but AIC returns 2 for seed 1
  and 6 for seed 6. AIC is the criterion that moves.
- **Changing $N$:** with seed 1, AIC picks 3 at $N=50$, 2 at $N=100$, and settles on the correct
  1 from $N=200$ onwards. More data disciplines the over-selection but, as question 5 shows,
  does not eliminate it.

The lesson for your write-up: never justify a lag order from one realisation.

In [12]:
# Question 5: how often does each criterion actually recover the TRUE lag order?
# We repeat the experiment over many seeds and count. The helper below is exactly the
# computation inside pfind, stripped of the DataFrame construction so 2000 fits are quick.

def optimal_lags(y, pmax):
    t, K = y.shape
    X = np.ones((1, t - pmax))
    for i in range(1, pmax + 1):
        X = np.vstack([X, y[pmax - i:t - i, :].T])
    Y = y[pmax:t, :].T
    T = t - pmax
    crit = np.zeros((pmax + 1, 3))
    for j in range(pmax + 1):
        Xj = X[:j * K + 1, :]
        B = Y @ Xj.T @ np.linalg.inv(Xj @ Xj.T)
        SIGMA = (Y - B @ Xj) @ (Y - B @ Xj).T / T
        ld = np.log(np.linalg.det(SIGMA))
        pen = j * K ** 2 + K
        crit[j] = [ld + np.log(T) / T * pen,
                   ld + 2 * np.log(np.log(T)) / T * pen,
                   ld + 2 / T * pen]
    return crit.argmin(axis=0)          # SIC, HQ, AIC

REPS = 500
rows = []
for p_true in (1, 2):
    for N in (100, 500):
        picks = np.array([optimal_lags(simulate(s, N, p_true), 8)
                          for s in range(1, REPS + 1)])
        rows.append([p_true, N] + [round(100 * (picks[:, k] == p_true).mean(), 1)
                                   for k in range(3)])

pd.DataFrame(rows, columns=["true p", "T", "SIC % correct", "HQ % correct", "AIC % correct"])

,true p,T,SIC % correct,HQ % correct,AIC % correct
0,1,100,99.6000,97.2000,85.4000
1,1,500,100.0000,98.8000,86.0000
2,2,100,2.8000,12.4000,26.2000
3,2,500,45.0000,82.8000,86.6000


**Answer to question 5.** Kilian and Lütkepohl (pp. 58-60) make two claims, and the table above
shows both.

*SIC and HQ are consistent; AIC is not.* When the truth is a VAR(1), SIC recovers it 99.6% of the
time at $T=100$ and 100% at $T=500$. AIC sits at roughly **85% and does not improve with more
data** — it over-selects with positive probability no matter how large the sample. That is what
"AIC is not consistent" means in practice.

*But consistency is an asymptotic promise, and small samples are a different country.* When the
truth is a VAR(2) whose second lag is weak (0.15), at $T=100$ SIC finds it only **2.8%** of the
time — it almost always under-selects to 1 lag. HQ manages 12.4%, AIC 26.2%. The criterion that
was best in the first two rows is the worst here, and the ranking only reverses once $T=500$.

This asymmetry matters. Over-selecting costs efficiency: you estimate coefficients that are
really zero. Under-selecting costs consistency: the omitted lags go into the residual, which
becomes serially correlated, and everything downstream is then misspecified. In a short sample
the "safe" parsimonious criterion is the one most likely to make the more dangerous error, which
is why you check the residuals rather than trusting the criterion.

In [13]:
np.random.seed(1)

# Parameters
N = 100  # Number of observations
rho1 = np.array([[0.8, 0],  # Autoregressive coefficient for lag 1
                 [0, 0.8]])

rho2 = np.array([[0.15, 0],  # Autoregressive coefficient for lag 2
                 [0, 0.15]])

alpha = np.array([0, 0])  # Mean of the VAR process

sigma = np.array([[1, 0],  # Covariance matrix of white noise
                  [0, 1]])

# Pre-allocate the array for the VAR(2) process
y = np.zeros((2, N))

# Generate white noise
epsilon = np.random.multivariate_normal([0, 0], sigma, N).T

# Generate the VAR(2) process
for t in range(2, N):
    y[:, t] = alpha + rho1 @ y[:, t-1] + rho2 @ y[:, t-2] + epsilon[:, t]

y = y.T

LagInformationValue, OptimalLag = pfind(y,12)

OptimalLag

,SIC,HQ,AIC
0,2,2,2


In [14]:
LagInformationValue

,Lag,SIC,HQ,AIC
0,0.0000,3.0956,3.0620,3.0393
1,1.0000,-0.1031,-0.2040,-0.2720
2,2.0000,-0.1643,-0.3324,-0.4458
3,3.0000,0.0268,-0.2085,-0.3673
4,4.0000,0.1821,-0.1205,-0.3247
5,5.0000,0.3301,-0.0397,-0.2892
6,6.0000,0.4845,0.0474,-0.2475
7,7.0000,0.6694,0.1651,-0.1751
8,8.0000,0.7964,0.2248,-0.1608
9,9.0000,0.9762,0.3375,-0.0935


In [15]:
OptimalLag

,SIC,HQ,AIC
0,2,2,2


In [16]:
# Is seed 1 representative? Re-run the VAR(2) experiment across the first 20 seeds
# and look at what SIC picks each time.
sic_picks = [int(pfind(simulate(s, 100, p_true=2), 12)[1]["SIC"].iloc[0]) for s in range(1, 21)]
print("SIC's pick for seeds 1 to 20:", sic_picks)
print(f"recovered the true p = 2 in {sum(p == 2 for p in sic_picks)} of 20 seeds")

SIC's pick for seeds 1 to 20: [2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
recovered the true p = 2 in 1 of 20 seeds


**Answer to question 6.** With the second lag added, all three criteria select **2** — the correct
order. That looks like a clean success, but check it against question 4 before you believe it.

The cell above re-runs the same experiment across the first twenty seeds: **seed 1 is the only one
that recovers $p = 2$ under SIC.** It is the lucky draw, and it is the seed the exercise happens to
ship with. This is the 2.8% from question 5 showing up concretely — with $\\rho_2 = 0.15$ and only
100 observations the second lag is too weak to be detected reliably. Had the exercise used seed 2,
the same code would have reported $p = 1$.

Instead of calculating the information criteria, one can perform a top-down sequential test. As noted by Kilian and Lütkepohl (2017), p. 51:

One possible sequential procedure involves testing the sequence of null hypotheses:
$H_{0}: A_{p_{\text{max}}} = 0$ vs. $H_{1}: A_{p_{\text{max}}} \neq 0$, 
$H_{0}: A_{p_{\text{max}}-1} = 0$ vs. $H_{1}: A_{p_{\text{max}}-1} \neq 0$, etc., 
until $H_{0}: A_{1} = 0$ vs. $H_{1}: A_{1} \neq 0$. The procedure continues, 
as long as the null hypothesis is not rejected. If there is a rejection, 
the testing procedure terminates, and we conclude that we require as many lags 
as maintained under the last alternative hypothesis. If none of the null hypotheses 
can be rejected, we conclude that $p = 0$. Such procedures are known as 
top-down or general-to-specific procedures because they start with the largest model 
and sequentially reduce the lag order of the model. In implementing this sequential test, 
the usual Wald or LR tests for parameter restrictions can be employed. 
For example, the LR statistic for testing a VAR($m$) against a VAR($m + 1$) has the form

\begin{equation*}
LR(m) = T \left[ \log(\det(\widetilde{\Sigma}_{u}(m))) - \log(\det(\widetilde{\Sigma}_{u}(m+1))) \right]
\end{equation*}

where

\begin{equation*}
\widetilde{\Sigma}_{u}(m) = T^{-1} \sum_{t=1}^{T} \hat{u}_{t} \hat{u}_{t}'
\end{equation*}

is the ML estimator of the residual covariance matrix for a VAR($m$) model. 
One also could alternatively use the LS residuals $\hat{u}_{t}$. 
If $H_{0}: A_{m+1} = 0$ is tested against $H_{1}: A_{m+1} \neq 0$, 
the $LR(m)$ statistic has an asymptotic $\chi^{2}(K^{2})$ distribution under $H_{0}$.


7. Use the following code to perform a top-down testing sequence. What is the optimal number of lags? Is the result the same as when using the Information criteria?

In [17]:
from scipy.stats import chi2

pmax = 12  # Set maximum lag order

def top_down(y, pmax):
    t, K = y.shape
    
    # Construct regressor matrix and dependent variable
    XMAX = np.ones((1, t - pmax))
    for i in range(1, pmax + 1):
        XMAX = np.vstack([XMAX, y[pmax - i:t - i, :].T])

    Y = y[pmax:t, :].T

    # Initialize Log Likelihood storage
    LL = []

    # Evaluate models for lag lengths from pmax down to 0
    for j in range(pmax, -1, -1):
        T = t - pmax
        X = XMAX[:j * K + 1, :]

        B = Y @ X.T @ np.linalg.inv(X @ X.T)
        SIGMA = (Y - B @ X) @ (Y - B @ X).T / T

        # Log-Likelihood
        LL.insert(0, [j, T * np.log(np.linalg.det(SIGMA))])

    LL = np.array(LL)

    # Initialize results table
    results = np.array([[0, LL[0, 1], 0, 0]])

    # Calculate LR test statistics and p-values
    for ii in range(pmax):
        lr_stat = LL[ii, 1] - LL[ii + 1, 1]
        p_value = 1 - chi2.cdf(lr_stat, K ** 2)
        results = np.vstack([results, [LL[ii + 1, 0], LL[ii + 1, 1], lr_stat, p_value]])

    # Convert results to a table
    table_lr = pd.DataFrame(results, columns=['Lag', 'Log. Likelihood', 'LR test', 'p-value'])

    return table_lr

top_down(y, pmax)


,Lag,Log. Likelihood,LR test,p-value
0,0.0000,263.4615,0.0000,0.0000
1,1.0000,-35.9395,299.4010,0.0000
2,2.0000,-59.2286,23.2891,0.0001
3,3.0000,-60.3234,1.0948,0.8951
4,4.0000,-64.5715,4.2481,0.3735
5,5.0000,-69.4495,4.8780,0.3000
6,6.0000,-73.7781,4.3286,0.3634
7,7.0000,-75.4100,1.6319,0.8030
8,8.0000,-82.1506,6.7406,0.1502
9,9.0000,-84.2290,2.0784,0.7213


**Answer to question 7.** On the VAR(2) data with `pmax = 12` the sequence fails to reject at every
lag from 12 down to 3, then rejects $H_0: A_2 = 0$ with $LR = 23.29$ and $p = 0.0001$. It stops
there and returns **$p = 2$**, agreeing with the information criteria.

Run the same procedure on the original VAR(1) data, though, and it rejects $H_0: A_2 = 0$ with
$LR = 10.18$, $p = 0.0375$ — so it returns $p = 2$ when the truth is 1. A marginal rejection just
above the 5% line is exactly what you expect to happen some of the time when you run a sequence of
tests, and it is a reminder that the top-down procedure has no more claim to being right than the
criteria do. Report both, and say which you chose and why.

8. Currently, the code does not take into account a linear trend in the data. Account for this by adding a linear trend using the following function and change the code accordingly:

In [18]:
# Question 8: account for a linear trend.
# The key idea: deterministic regressors are the same for EVERY candidate lag order, so they
# belong at the TOP of the regressor matrix. The lag block is then sliced in below them.

def pfind_trend(y, pmax, trend=True):
    t, K = y.shape

    DET = np.ones((1, t - pmax))                                  # constant
    if trend:
        DET = np.vstack([DET, np.arange(1, t - pmax + 1).reshape(1, -1)])
    nd = DET.shape[0]                                             # size of the fixed block

    XMAX = DET.copy()
    for i in range(1, pmax + 1):
        XMAX = np.vstack([XMAX, y[pmax - i:t - i, :].T])

    Y = y[pmax:t, :].T
    T = t - pmax

    crit = np.zeros((pmax + 1, 3))
    for j in range(pmax + 1):
        X = XMAX[:j * K + nd, :]                                  # note: + nd, not + 1
        B = Y @ X.T @ np.linalg.inv(X @ X.T)
        SIGMA = (Y - B @ X) @ (Y - B @ X).T / T
        ld = np.log(np.linalg.det(SIGMA))
        pen = j * K ** 2 + K
        crit[j] = [ld + np.log(T) / T * pen,
                   ld + 2 * np.log(np.log(T)) / T * pen,
                   ld + 2 / T * pen]

    table = pd.DataFrame(np.column_stack([np.arange(pmax + 1), crit]),
                         columns=["Lag", "SIC", "HQ", "AIC"])
    optimal = pd.DataFrame([crit.argmin(axis=0)], columns=["SIC", "HQ", "AIC"])
    return table, optimal


rows = []
for name, series in [("VAR(1) data", simulate(1, 100, 1)), ("VAR(2) data", simulate(1, 100, 2))]:
    for label, use_trend in [("no trend", False), ("with trend", True)]:
        opt = pfind_trend(series, 12, trend=use_trend)[1]
        rows.append([name, label] + opt.iloc[0].tolist())

pd.DataFrame(rows, columns=["data", "specification", "SIC", "HQ", "AIC"])

,data,specification,SIC,HQ,AIC
0,VAR(1) data,no trend,1,1,2
1,VAR(1) data,with trend,1,1,2
2,VAR(2) data,no trend,2,2,2
3,VAR(2) data,with trend,2,2,2


9. Did the results change?

10. At last, use the code you have written to answer **questions 2 and 3 of Homework 1**.

The homework VAR is not quite the model you have been simulating. From the assignment
hints, the system is

$$x = [\\,rfor,\\ lgdp,\\ pi,\\ dlrexc,\\ rdom\\,]$$

estimated on 1983Q1-2004Q4 together with a constant, a **linear trend**, and the dummies
`du93Q1`, `du92Q3` and `du95Q4`.

So the regressor matrix has two parts: a deterministic block that is the *same* for every
candidate lag order, and a lag block that grows with $m$. Only the second part changes as
you move down the table.

- Load the `result.csv` you produced in Exercise Class 1
- Extend `pfind` (and `top_down`) so the deterministic block is `[constant, dummies, trend]`
- Use `pmax = 4`: this is quarterly data, so four lags is one year
- What do the three criteria say? What does the top-down sequence say? Do they agree,
  and does the answer change if you set `pmax = 8` or `12`?

**Answer to question 9.** No — the selected lag order is unchanged on both simulated series.

That is the *right* answer here, and worth understanding rather than just reporting: the DGP in
this exercise has no trend, so the estimated trend coefficient is just noise. It costs one
degree of freedom per equation and shifts $\\log\\det(\\widetilde{\\Sigma}_u)$ very
slightly, but not nearly enough to move the minimum.

It will matter on the homework data. There `lgdp` trends at roughly 2% per year, and Bjørnland
(2009) deliberately includes a linear trend in the VAR rather than detrending output beforehand.
Omitting it there does not just cost precision — it leaves a trend in the residuals, and the lag
order would then be selected to soak up that trend.

In [19]:
from pathlib import Path

# result.csv is the file produced in Exercise Class 1
CANDIDATES = [
    Path.cwd() / "result.csv",
    Path.cwd().parent / "Exercise Class 1" / "result.csv",
]
CSV = next((p for p in CANDIDATES if p.exists()), None)
if CSV is None:
    raise FileNotFoundError(
        "Could not find result.csv - run the Exercise Class 1 notebook first. Looked in:\n  "
        + "\n  ".join(str(p) for p in CANDIDATES)
    )

data = pd.read_csv(CSV)

ENDOG = ["rfor", "lgdp", "pi", "dlrexc", "rdom"]   # order matters, see the hints
DUMMIES = ["du93Q1", "du92Q3", "du95Q4"]

y = data[ENDOG].to_numpy()
dum = data[DUMMIES].to_numpy(float)
print(f"T = {y.shape[0]} observations, K = {y.shape[1]} variables")


def build_regressors(y, pmax, dum=None, trend=True):
    """Deterministic block (same for every m) stacked on top of the lag block."""
    t, K = y.shape
    blocks = [np.ones((1, t - pmax))]                       # constant
    if dum is not None:
        blocks.append(dum[pmax:t, :].T)                     # dummies
    if trend:
        blocks.append(np.arange(1, t - pmax + 1).reshape(1, -1))  # linear trend
    DET = np.vstack(blocks)
    LAGS = np.vstack([y[pmax - i:t - i, :].T for i in range(1, pmax + 1)])
    return DET, LAGS, y[pmax:t, :].T


def pfind_det(y, pmax, dum=None, trend=True):
    """pfind, but with a deterministic block that does not change with m."""
    t, K = y.shape
    DET, LAGS, Y = build_regressors(y, pmax, dum, trend)
    T = t - pmax
    rows, logdet = [], []
    for m in range(pmax + 1):
        X = np.vstack([DET, LAGS[:m * K, :]]) if m else DET
        B = Y @ X.T @ np.linalg.inv(X @ X.T)
        SIGMA = (Y - B @ X) @ (Y - B @ X).T / T
        ld = np.log(np.linalg.det(SIGMA)); logdet.append(ld)
        pen = m * K ** 2 + K
        rows.append([m,
                     ld + np.log(T) / T * pen,
                     ld + 2 * np.log(np.log(T)) / T * pen,
                     ld + 2 / T * pen])
    table = pd.DataFrame(rows, columns=["Lag", "SIC", "HQ", "AIC"])
    optimal = pd.DataFrame([[int(table[c].idxmin()) for c in ("SIC", "HQ", "AIC")]],
                           columns=["SIC", "HQ", "AIC"])
    return table, optimal, np.array(logdet), T


# LR sequence, reusing the log-determinants from pfind_det
def sequence(y, pmax, dum=None, trend=True, alpha=0.05):
    from scipy.stats import chi2
    t, K = y.shape
    _, _, logdet, T = pfind_det(y, pmax, dum, trend)
    rows = []
    for m in range(pmax, 0, -1):
        stat = T * (logdet[m - 1] - logdet[m])
        pval = 1 - chi2.cdf(stat, K ** 2)
        rows.append([m, stat, pval, "reject" if pval < alpha else "fail to reject"])
        if pval < alpha:
            break
    return pd.DataFrame(rows, columns=["H0: A_m = 0", "LR", "p-value", "decision"])


for pmax in (4, 8, 12):
    table, optimal, _, _ = pfind_det(y, pmax, dum, trend=True)
    print(f"\npmax = {pmax}: optimal lag ->", optimal.to_dict("records")[0])

print("\nInformation criteria, pmax = 4:")
table, optimal, _, _ = pfind_det(y, 4, dum, trend=True)
print(table.round(4).to_string(index=False))

print("\nTop-down sequence, pmax = 4:")
print(sequence(y, 4, dum, trend=True).to_string(index=False))

T = 88 observations, K = 5 variables

pmax = 4: optimal lag -> {'SIC': 1, 'HQ': 1, 'AIC': 3}

pmax = 8: optimal lag -> {'SIC': 1, 'HQ': 1, 'AIC': 8}

pmax = 12: optimal lag -> {'SIC': 1, 'HQ': 12, 'AIC': 12}

Information criteria, pmax = 4:
 Lag      SIC       HQ      AIC
   0 -22.0929 -22.1794 -22.2376
   1 -28.8463 -29.3655 -29.7145
   2 -28.0719 -29.0237 -29.6635
   3 -27.4260 -28.8105 -29.7411
   4 -26.3783 -28.1953 -29.4168

Top-down sequence, pmax = 4:
 H0: A_m = 0      LR  p-value       decision
           4 22.7603   0.5916 fail to reject
           3 56.5208   0.0003         reject


**What you should find.** With `pmax = 4` the AIC and the top-down sequence both select
**3 lags**, which is the lag order the assignment hints give. SIC and HQ select 1.

If you set `pmax = 8` or `12` instead, AIC and the top-down sequence run all the way to the
boundary. A criterion that selects `pmax` has not found a minimum, it has run out of
degrees of freedom: at 12 lags this system estimates 305 parameters from 88 observations.

For question 3, report the comparison rather than a single number: the disagreement between
the criteria, and the sensitivity to `pmax`, is the actual answer.